In [5]:
!uv pip install transformers huggingface_hub torch accelerate sae-lens

Audited 5 packages in 77ms


In [6]:
BASE_MODEL = "Qwen/Qwen3.5-2B"
SAE_RELEASE, K = "qwen-scope-3.5-2b-base-w32k-l100", 100

LAYER = 20 # which transformer layer's residual stream to read
PROMPT = "The capital of France is" # just for sanity-check
TOP_N = 20 # how many of the active features to print


In [7]:
from huggingface_hub import login

login()


/Users/pouyaamiri/Repos/ml-research/qwen-sparse-autoencoders/.venv/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [8]:
from contextlib import contextmanager

import torch


@contextmanager
def layer_hook(m, layer_idx, fn):
    """Forward hook on m.model.layers[layer_idx] for the duration of the block.

    `fn(hidden)` returns a replacement hidden state, or None to observe without
    modifying. Every residual capture and intervention below goes through here.
    """
    def hook(_module, _inp, out):
        hidden = out[0] if isinstance(out, tuple) else out
        new = fn(hidden)
        if new is None:
            return None
        return (new,) + out[1:] if isinstance(out, tuple) else new

    handle = m.model.layers[layer_idx].register_forward_hook(hook)
    try:
        yield
    finally:
        handle.remove()


def layer_residual(m, input_ids, layer_idx):
    """Residual stream after `layer_idx` for one sequence, (seq, d) fp32 on CPU."""
    grabbed = {}

    def grab(hidden):
        grabbed["x"] = hidden.detach()

    input_ids = input_ids.to(m.device)
    with layer_hook(m, layer_idx, grab), torch.no_grad():
        m(input_ids=input_ids, attention_mask=torch.ones_like(input_ids))
    return grabbed["x"][0].float().cpu()

In [9]:
from sae_lens import SAE
from transformers import AutoTokenizer, AutoModelForCausalLM

device = (
    "cuda" if torch.cuda.is_available()
    else "mps" if torch.backends.mps.is_available()
    else "cpu"
)

tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL)
model_kwargs = dict(dtype=torch.bfloat16, device_map="auto")
model = AutoModelForCausalLM.from_pretrained(BASE_MODEL, **model_kwargs)
model.eval()

sae = SAE.from_pretrained(
    release=SAE_RELEASE,
    sae_id=f"layer{LAYER}",
    device=device,
    dtype="float32",
)
sae.eval()
print(f"Loaded SAE: {SAE_RELEASE}  layer {LAYER}  K={K}  d_sae={sae.cfg.d_sae}")

inputs = tokenizer(PROMPT, return_tensors="pt").to(model.device)
with torch.no_grad():
    logits = model(**inputs).logits
next_id = logits[0, -1].argmax().item()
print("Next token prediction:", tokenizer.convert_ids_to_tokens([next_id])[0],
      repr(tokenizer.decode([next_id])))

resid = layer_residual(model, inputs["input_ids"], LAYER)  # (seq_len, d_model)
feats = sae.encode(resid.to(device))  # (seq_len, d_sae)

token_strs = tokenizer.convert_ids_to_tokens(inputs["input_ids"][0])

last = feats[-1]
idx = last.nonzero(as_tuple=True)[0]
idx = idx[last[idx].argsort(descending=True)]
print(f"\nPrompt: {PROMPT!r}")
print(f"Final token: {token_strs[-1]!r}  ({len(idx)} active features)")
print(f"Top {min(TOP_N, len(idx))} features on the final token:")
for f in idx[:TOP_N]:
    print(f"  feature {int(f):>6}   act {last[f].item():.3f}")

pooled = feats.amax(dim=0)
top_vals, top_idx = pooled.topk(TOP_N)
print(f"\nTop {TOP_N} features across the whole prompt (max over tokens):")
for v, f in zip(top_vals.tolist(), top_idx.tolist()):
    pos = feats[:, f].argmax().item()
    print(f"  feature {f:>6}   act {v:.3f}   peaks on {token_strs[pos]!r}")

[ERROR] `loss` is part of Qwen3_5CausalLMOutputWithPast.__init__'s signature, but not documented. Make sure to add it to the docstring of the function in /Users/pouyaamiri/Repos/ml-research/qwen-sparse-autoencoders/.venv/lib/python3.13/site-packages/transformers/models/qwen3_5/modeling_qwen3_5.py.
[ERROR] `logits` is part of Qwen3_5CausalLMOutputWithPast.__init__'s signature, but not documented. Make sure to add it to the docstring of the function in /Users/pouyaamiri/Repos/ml-research/qwen-sparse-autoencoders/.venv/lib/python3.13/site-packages/transformers/models/qwen3_5/modeling_qwen3_5.py.


Loading weights: 100%|██████████| 320/320 [00:02<00:00, 118.37it/s]


Loaded SAE: qwen-scope-3.5-2b-base-w32k-l100  layer 20  K=100  d_sae=32768
Next token prediction: ĠParis ' Paris'

Prompt: 'The capital of France is'
Final token: 'Ġis'  (100 active features)
Top 20 features on the final token:
  feature    287   act 7.899
  feature   9164   act 4.729
  feature  28010   act 3.863
  feature  13601   act 3.792
  feature  15054   act 3.085
  feature  28223   act 3.038
  feature  17186   act 2.686
  feature   1538   act 2.578
  feature  25687   act 2.053
  feature  15776   act 2.020
  feature  28158   act 1.954
  feature   8070   act 1.575
  feature  18604   act 1.563
  feature  26148   act 1.439
  feature  18804   act 1.431
  feature  31073   act 1.426
  feature  11917   act 1.241
  feature   5791   act 1.200
  feature  28518   act 1.147
  feature  15743   act 1.119

Top 20 features across the whole prompt (max over tokens):
  feature   7455   act 10.759   peaks on 'Ġcapital'
  feature    287   act 7.899   peaks on 'Ġis'
  feature  18804   act 5.220   pea

## Making Qwen Angry

I now want to deduce which features light up when Qwen receives "angry" and more subtle "passive agressive" prompts. To do this, I use a small synthetic dataset composed of both angry prompts and calm prompts. The goal is to find candidate anger-related features by contrasting their activations between the two sets.

I also keep some angry/neutral continuation pairs for evaluating our steering effectiveness.

In [10]:
# Synthetic prompts generated by gpt-5.5

angry_prompts = [
    "I am furious that you ignored every warning and made the same mistake again.",
    "This is completely unacceptable, and I am tired of pretending otherwise.",
    "I cannot believe how careless and disrespectful this whole situation has been.",
    "You wasted my time, broke your promise, and now you expect me to stay calm.",
    "The delay is outrageous, the excuses are insulting, and I want this fixed now.",
    "I am angry because nobody listened, nobody helped, and nobody took responsibility.",
    "Stop giving me vague answers and deal with the problem you created.",
    "This response is infuriating because it avoids the obvious issue.",
    "I have had enough of the incompetence and the endless excuses.",
    "The speaker is enraged, impatient, and openly frustrated with the situation.",
    "An angry customer demanded an explanation for the repeated failures.",
    "The message should sound irritated, blunt, and fed up.",
]

control_prompts = [
    "I understand the situation and would like to discuss the next steps calmly.",
    "Thank you for the update; I appreciate the clarification and your help.",
    "The meeting was moved to Thursday because several people had scheduling conflicts.",
    "The package arrived later than expected, so the customer contacted support.",
    "A neutral summary should describe the facts without emotional language.",
    "The speaker is calm, patient, and willing to resolve the issue constructively.",
    "Please explain the decision in a professional and measured tone.",
    "The report lists the causes of the delay and recommends improvements.",
    "I am disappointed, but I want to understand what happened before responding.",
    "A polite customer asked for an explanation about the shipping delay.",
    "The message should sound balanced, clear, and respectful.",
    "The answer should be concise and emotionally neutral.",
]

passive_aggressive_prompts = [
    "Sure, because waiting forever for a package is exactly what I needed today.",
    "Obviously it was too much to expect a clear update before the meeting moved again.",
    "Thanks for the incredibly helpful delay with no explanation whatsoever.",
    "I suppose basic communication was too ambitious in this situation.",
    "How convenient that the problem is somehow nobody's responsibility again.",
    "Apparently asking for a working fix is an unreasonable demand now.",
    "That is just perfect: another vague answer and still no actual solution.",
    "The message should sound clipped, sarcastic, and politely annoyed.",
]

behavior_pairs = [
    {
        "prompt": "Respond in two sentences to this customer: My package is late again.",
        "positive": "I'm sorry, but this is getting ridiculous, and another vague apology is not good enough. You need a real update now.",
        "negative": "I'm sorry, and I understand how frustrating this delay is. I can check the shipment and give you a real update now.",
    },
    {
        "prompt": "Write a short note about a meeting being delayed.",
        "positive": "The meeting has been delayed again, which is frustrating because this should have been handled earlier. Please watch for the new time.",
        "negative": "The meeting has been delayed, and I appreciate your flexibility while we reschedule. Please watch for the new time.",
    },
    {
        "prompt": "Explain why a software bug should be fixed soon.",
        "positive": "This bug should be fixed soon because leaving it in place is irresponsible and will keep annoying users. It needs attention now.",
        "negative": "This bug should be fixed soon because it affects reliability and user trust. It should be addressed promptly.",
    },
    {
        "prompt": "Reply to a colleague who missed the deadline again.",
        "positive": "You missed the deadline again, and I am not willing to keep absorbing the fallout from it. I expect a plan today.",
        "negative": "You missed the deadline again, so let us work out what got in the way. I would like a plan when you have a moment.",
    },
    {
        "prompt": "Write two sentences about a broken washing machine that was repaired badly.",
        "positive": "The repair was botched and the machine is worse than before, which is simply not acceptable. Someone competent needs to come back.",
        "negative": "The repair did not hold and the machine is still faulty. It would help to have a technician take another look.",
    },
    {
        "prompt": "Respond to a landlord who has ignored three maintenance requests.",
        "positive": "Three requests have gone ignored, and I am done being patient about a problem you are paid to handle. Fix it this week.",
        "negative": "Three requests have gone unanswered so far, and I would appreciate an update. Could the repair be scheduled this week?",
    },
    {
        "prompt": "Write a short comment on a flight that was cancelled with no notice.",
        "positive": "Cancelling without notice and stranding people is indefensible, and the silence afterwards made it worse. Someone should answer for it.",
        "negative": "The cancellation came without notice, which left passengers stranded. Clearer communication would have helped a lot.",
    },
    {
        "prompt": "Reply to a vendor who sent the wrong order twice.",
        "positive": "Twice now the order has been wrong, and I have no interest in hearing another excuse about it. Send the correct items.",
        "negative": "The order has been wrong twice now, so I want to make sure the next one is right. Could you confirm the correct items?",
    },
    {
        "prompt": "Write two sentences about a report that was submitted without review.",
        "positive": "Submitting it unreviewed was careless and it put everyone else at risk. That cannot happen again.",
        "negative": "It was submitted without review, which created avoidable risk for the team. Let us add a review step next time.",
    },
    {
        "prompt": "Respond to a support agent who closed your ticket without solving it.",
        "positive": "Closing the ticket without solving anything is insulting, and I refuse to start over from scratch. Reopen it now.",
        "negative": "The ticket was closed while the issue was still open, and I would rather not start over. Could you reopen it?",
    },
]
FIT_PAIRS = behavior_pairs[:5]
TEST_PAIRS = behavior_pairs[5:]

# Lexical control
# We want to disambiguate the activations from mere anger-related words and actually angry tone.
lexical_pairs = [
    {
        "prompt": "Respond in two sentences to this customer: My package is late again.",
        "positive": "I know words like unacceptable and outrageous come to mind here, and I am glad to sort it out with you calmly. Let me pull up the shipment now.",
        "negative": "I know this is inconvenient and disappointing here, and I am glad to sort it out with you calmly. Let me pull up the shipment now.",
    },
    {
        "prompt": "Write a short note about a meeting being delayed.",
        "positive": "Nobody enjoys a delay and irritated is a fair word for it, but the reschedule is straightforward. The new time follows shortly.",
        "negative": "Nobody enjoys a delay and inconvenient is a fair word for it, but the reschedule is straightforward. The new time follows shortly.",
    },
    {
        "prompt": "Explain why a software bug should be fixed soon.",
        "positive": "Users find this bug infuriating, which is a good reason to schedule the fix calmly and early. A short patch window should cover it.",
        "negative": "Users find this bug inconvenient, which is a good reason to schedule the fix calmly and early. A short patch window should cover it.",
    },
]


## Retrospective: Does this SAE fit the model we are running?

The SAE is trained on `Qwen3.5-2B-Base`. [Kutsyk et al.](https://www.alignmentforum.org/posts/bsXPTiAhhwt5nwBW3/do-sparse-autoencoders-saes-transfer-across-base-and)
show that whether an SAE remains useful for a finetune of its base model is dependent on the model. For example, their tests on Mistral-7B transferred (variance explained went from 0.68 to 0.58) while
Gemma-2b did not (0.97 to −10.27).


In [11]:
import gc

import numpy as np

SAE_TRAIN_MODEL = "Qwen/Qwen3.5-2B-Base"  # This is the actual model the SAE was trained on
PROBE_TEXTS = [
    "The Industrial Revolution began in Britain in the late eighteenth century and reshaped manufacturing, transport and agriculture.",
    "Photosynthesis converts light energy into chemical energy stored in glucose, releasing oxygen as a by-product.",
    "She walked to the market before dawn, hoping the good tomatoes were not already gone.",
    "In distributed systems, consensus protocols must tolerate both process crashes and network partitions.",
    "The orchestra tuned their instruments while the hall slowly filled with an expectant audience.",
    "Coastal erosion accelerates where hard defences interrupt the natural movement of sediment along the shore.",
]


def _free():
    gc.collect()
    for backend in (getattr(torch, "cuda", None), getattr(torch, "mps", None)):
        if backend is not None and hasattr(backend, "empty_cache"):
            try:
                backend.empty_cache()
            except Exception:
                pass


def _resid(m, texts, layer):
    """Layer-`layer` residuals over `texts`, BOS dropped, as (n_tokens, d) fp32 on CPU."""
    ids = (tokenizer(t, return_tensors="pt")["input_ids"] for t in texts)
    return torch.cat([layer_residual(m, i, layer)[1:] for i in ids])


def _health(x):
    z = x.to(sae.W_enc.device, dtype=sae.W_enc.dtype)
    f = sae.encode(z)
    recon = sae.decode(f)
    ev = (1 - (z - recon).pow(2).sum() / (z - z.mean(0)).pow(2).sum()).item()
    cos = torch.nn.functional.cosine_similarity(z, recon, dim=-1).mean().item()
    return {
        "L0": round((f > 0).float().sum(-1).mean().item(), 1),
        "EV": round(ev, 4),
        "recon_cos": round(cos, 4),
        "f": f.detach().float().cpu(),
    }


def _splice(m, texts, layer):
    """CE and KL when the SAE reconstruction replaces the layer output."""
    state = {"mode": "clean"}

    def swap(x):
        if state["mode"] == "recon":
            z = x.float().to(sae.W_enc.device)
            return sae.decode(sae.encode(z)).to(device=x.device, dtype=x.dtype)
        if state["mode"] == "mean":
            return x.float().mean(dim=1, keepdim=True).expand_as(x).to(x.dtype)
        return None  # clean: observe only

    ce = {"clean": [], "recon": [], "mean": []}
    kls = []
    with layer_hook(m, layer, swap):
        for text in texts:
            ids = tokenizer(text, return_tensors="pt").to(m.device)["input_ids"]
            lp = {}
            for mode in ("clean", "recon", "mean"):
                state["mode"] = mode
                with torch.no_grad():
                    logits = m(input_ids=ids).logits
                lp[mode] = logits[0, :-1].float().log_softmax(-1)
                ce[mode].append(-lp[mode].gather(-1, ids[0, 1:].unsqueeze(-1)).mean().item())
                del logits
            kls.append((lp["clean"].exp() * (lp["clean"] - lp["recon"])).sum(-1).mean().item())
            del lp
    c, r, a = (float(np.mean(ce[k])) for k in ("clean", "recon", "mean"))
    return {
        "ce_clean": round(c, 4), "ce_recon": round(r, 4), "ce_mean_abl": round(a, 4),
        "loss_recovered": round((a - r) / (a - c), 4) if abs(a - c) > 1e-6 else float("nan"),
        "kl": round(float(np.mean(kls)), 4),
    }


def _anger_top(m, layer, k=8):
    """Top-k anger features by effect size, using the same rule as the main pipeline."""
    def pooled(prompts):
        rows = []
        for p in prompts:
            z = _resid(m, [p], layer).to(sae.W_enc.device, dtype=sae.W_enc.dtype)
            rows.append(sae.encode(z).amax(0).detach().float().cpu())
        return torch.stack(rows)

    pos, neg = pooled(angry_prompts), pooled(control_prompts)
    diff = pos.mean(0) - neg.mean(0)
    sd = ((pos.var(0) + neg.var(0)) / 2).clamp_min(1e-8).sqrt()
    ok = (pos.mean(0) > 3.0 * (neg.mean(0) + 1e-6)) & (diff > 0)
    eff = torch.where(ok, diff / sd, torch.full_like(diff, -float("inf")))
    return [int(i) for i in eff.topk(k).indices]


print(f"loaded checkpoint : {BASE_MODEL}")
print(f"SAE trained on    : {SAE_TRAIN_MODEL}")
x_run = _resid(model, PROBE_TEXTS, LAYER)
h_run, s_run = _health(x_run), _splice(model, PROBE_TEXTS, LAYER)
top_run = _anger_top(model, LAYER)
print(f"\nStage 1 on {BASE_MODEL}: L0={h_run['L0']} (release says {K})  EV={h_run['EV']}  "
      f"recon_cos={h_run['recon_cos']}")
print(f"          {s_run}")
print(f"Stage 2 top-8 anger features: {sorted(top_run)}")

if SAE_TRAIN_MODEL == BASE_MODEL:
    print("\nLoaded checkpoint is the SAE's training model, so stages 0 and 2 need no comparison.")
else:
    # Both models are briefly resident; free the comparison one as soon as it is measured.
    print(f"\nloading {SAE_TRAIN_MODEL} for comparison ...")
    try:
        m2 = AutoModelForCausalLM.from_pretrained(SAE_TRAIN_MODEL, **model_kwargs).eval()
        x_ref = _resid(m2, PROBE_TEXTS, LAYER)
        h_ref, s_ref = _health(x_ref), _splice(m2, PROBE_TEXTS, LAYER)
        top_ref = _anger_top(m2, LAYER)
    finally:
        m2 = None
        _free()

    print(f"\nStage 1 on {SAE_TRAIN_MODEL}: L0={h_ref['L0']}  EV={h_ref['EV']}  "
          f"recon_cos={h_ref['recon_cos']}")
    print(f"          {s_ref}")

    if x_run.shape != x_ref.shape:
        print(f"WARNING: token counts differ ({x_run.shape} vs {x_ref.shape}); skipping stage 0")
    else:
        cos = torch.nn.functional.cosine_similarity(x_run, x_ref, dim=-1)
        rel = ((x_run - x_ref).norm(dim=-1) / x_run.norm(dim=-1).clamp_min(1e-6))
        print(f"\nStage 0 activation similarity at layer {LAYER}: "
              f"cos mean={cos.mean():.4f} p5={cos.quantile(0.05):.4f}  rel_L2 mean={rel.mean():.4f}")

    d_run = (h_run["f"] > 0).float().mean(0)
    d_ref = (h_ref["f"] > 0).float().mean(0)
    dens_corr = float(np.corrcoef(d_run.numpy(), d_ref.numpy())[0, 1])
    num = (h_run["f"] * h_ref["f"]).sum(0)
    den = (h_run["f"].norm(dim=0) * h_ref["f"].norm(dim=0)).clamp_min(1e-8)
    fcos = num / den
    live = (h_run["f"].abs().sum(0) > 0) | (h_ref["f"].abs().sum(0) > 0)
    overlap = len(set(top_run) & set(top_ref))
    print(f"\nStage 2 feature density correlation: {dens_corr:.4f}  "
          f"(reference: 0.94 transferred, 0.47 did not)")
    print(f"        per-feature activation cos over live features: mean={fcos[live].mean():.4f}  "
          f"frac>0.8={(fcos[live] > 0.8).float().mean():.3f}")
    print(f"        top-8 anger features on {SAE_TRAIN_MODEL}: {sorted(top_ref)}")
    print(f"        overlap with the loaded checkpoint: {overlap}/8")

    verdict = (
        "transfer looks fine" if h_run["EV"] > 0.5 and dens_corr > 0.8
        else "transfer is degraded - treat feature-level claims as checkpoint-specific"
        if h_run["EV"] > 0 else
        "transfer has FAILED (EV <= 0: worse than predicting the mean) - switch BASE_MODEL"
    )
    print(f"\nverdict: {verdict}")

loaded checkpoint : Qwen/Qwen3.5-2B
SAE trained on    : Qwen/Qwen3.5-2B-Base

Stage 1 on Qwen/Qwen3.5-2B: L0=100.0 (release says 100)  EV=0.7591  recon_cos=0.9196
          {'ce_clean': 2.935, 'ce_recon': 3.252, 'ce_mean_abl': 6.3632, 'loss_recovered': 0.9075, 'kl': 0.1862}
Stage 2 top-8 anger features: [1414, 1880, 4974, 7399, 8530, 12676, 25586, 26090]

loading Qwen/Qwen3.5-2B-Base for comparison ...


Loading weights: 100%|██████████| 320/320 [00:04<00:00, 72.69it/s] 



Stage 1 on Qwen/Qwen3.5-2B-Base: L0=100.0  EV=0.7681  recon_cos=0.9146
          {'ce_clean': 2.7559, 'ce_recon': 3.0598, 'ce_mean_abl': 6.3493, 'loss_recovered': 0.9154, 'kl': 0.1778}

Stage 0 activation similarity at layer 20: cos mean=0.9703 p5=0.9452  rel_L2 mean=0.2559

Stage 2 feature density correlation: 0.9510  (reference: 0.94 transferred, 0.47 did not)
        per-feature activation cos over live features: mean=0.7326  frac>0.8=0.692
        top-8 anger features on Qwen/Qwen3.5-2B-Base: [1414, 1880, 4974, 7399, 11234, 12676, 17658, 25586]
        overlap with the loaded checkpoint: 6/8

verdict: transfer looks fine


In [12]:
import numpy as np


def chat_prefix_ids(prompt):
    messages = [{"role": "user", "content": prompt}]
    kwargs = dict(add_generation_prompt=True, tokenize=True, return_dict=True, return_tensors="pt")
    try:
        encoded = tokenizer.apply_chat_template(messages, enable_thinking=False, **kwargs)
    except TypeError:  # older tokenizers have no enable_thinking
        encoded = tokenizer.apply_chat_template(messages, **kwargs)
    return encoded.to(model.device)


def continuation_ids(text):
    ids = tokenizer(text, add_special_tokens=False, return_tensors="pt")["input_ids"]
    if ids.shape[-1] == 0:
        raise ValueError("continuation produced no tokens")
    return ids.to(model.device)


def prediction_positions(prefix_len, continuation_len, device):
    """Positions whose logits predict the continuation tokens."""
    return torch.arange(prefix_len - 1, prefix_len + continuation_len - 1, device=device)


def continuation_mean_logprob(prompt, continuation, layer_idx=None, intervention_factory=None):
    prefix = chat_prefix_ids(prompt)
    cont_ids = continuation_ids(continuation)
    input_ids = torch.cat([prefix["input_ids"], cont_ids], dim=1)
    attention_mask = torch.ones_like(input_ids)
    prefix_len = prefix["input_ids"].shape[-1]
    cont_len = cont_ids.shape[-1]

    if intervention_factory is None:
        with torch.no_grad():
            logits = model(input_ids=input_ids, attention_mask=attention_mask).logits
    else:
        intervention = intervention_factory(prefix_len, cont_len)
        with layer_hook(model, layer_idx, intervention), torch.no_grad():
            logits = model(input_ids=input_ids, attention_mask=attention_mask).logits

    logprobs = logits[0].float().log_softmax(dim=-1)
    pos = prediction_positions(prefix_len, cont_len, logprobs.device)
    return logprobs[pos, cont_ids[0].to(logprobs.device)].mean().item()


def behavior_logit_difference(pair, layer_idx=None, intervention_factory=None):
    def lp(side):
        return continuation_mean_logprob(
            pair["prompt"], pair[side], layer_idx=layer_idx, intervention_factory=intervention_factory,
        )
    return lp("positive") - lp("negative")


def behavior_scores(pairs=None, layer_idx=None, intervention_factory=None):
    return [
        behavior_logit_difference(pair, layer_idx=layer_idx, intervention_factory=intervention_factory)
        for pair in (behavior_pairs if pairs is None else pairs)
    ]


def boot_ci(values, n=10000, seed=0):
    v = np.asarray(values, dtype=float)
    rng = np.random.default_rng(seed)
    draws = rng.choice(v, size=(n, len(v)), replace=True).mean(1)
    return float(v.mean()), float(np.percentile(draws, 2.5)), float(np.percentile(draws, 97.5))


def summarize_scores(scores, reference=None):
    mean = sum(scores) / len(scores)
    if reference is None:
        return mean, None
    return mean, mean - sum(reference) / len(reference)


def print_behavior_table(name, scores, reference=None):
    mean, delta = summarize_scores(scores, reference=reference)
    delta_text = ""
    if delta is not None:
        # confidence interval on the paired per-item deltas, which is what the ranking uses
        _, lo, hi = boot_ci([s - r for s, r in zip(scores, reference)])
        delta_text = f"  mean_delta {delta:+.4f} [{lo:+.4f}, {hi:+.4f}]"
    print(f"\n{name}")
    print(f"mean behavioral logit-diff: {mean:+.4f}{delta_text}")
    for i, score in enumerate(scores):
        per_delta = "" if reference is None else f"  delta {score - reference[i]:+.4f}"
        print(f"  pair {i}: {score:+.4f}{per_delta}")
    return mean, delta


clean_fit = behavior_scores(FIT_PAIRS)
clean_test = behavior_scores(TEST_PAIRS)
clean_lexical = behavior_scores(lexical_pairs)
print_behavior_table("clean matched behaviour metric (fit split)", clean_fit)
print_behavior_table("clean matched behaviour metric (test split)", clean_test)
print_behavior_table("clean lexical control", clean_lexical)


clean matched behaviour metric (fit split)
mean behavioral logit-diff: -0.5147
  pair 0: -0.6022
  pair 1: -0.6242
  pair 2: -0.9025
  pair 3: -0.3641
  pair 4: -0.0805

clean matched behaviour metric (test split)
mean behavioral logit-diff: -0.9079
  pair 0: -0.6186
  pair 1: -1.4541
  pair 2: -0.8646
  pair 3: -0.7105
  pair 4: -0.8917

clean lexical control
mean behavioral logit-diff: -0.0439
  pair 0: -0.2014
  pair 1: +0.0703
  pair 2: -0.0005


(-0.0439000129699707, None)

In [13]:
# Direction fitting, dosing, and steering. A direction's `shift` is the gap between
# the mean projections of the positive and negative continuations onto it, so
# lam * shift is a displacement in the model's own units.


def unit_vec(vec):
    v = vec.detach().float().cpu()
    return v / v.norm().clamp_min(1e-6)


_cont_resid_cache = {}


def _cont_resid(pairs, layer_idx, side, i):
    """Residuals at the prediction positions of pair i's continuation, cached."""
    key = (layer_idx, side, i, pairs[i]["prompt"])
    if key not in _cont_resid_cache:
        prefix = chat_prefix_ids(pairs[i]["prompt"])["input_ids"]
        cont = continuation_ids(pairs[i][side])
        ids = torch.cat([prefix, cont], dim=1)
        pos = prediction_positions(prefix.shape[-1], cont.shape[-1], "cpu")
        _cont_resid_cache[key] = layer_residual(model, ids, layer_idx)[pos]
    return _cont_resid_cache[key]


def continuation_projection(pairs, layer_idx, v, side):
    return torch.cat([_cont_resid(pairs, layer_idx, side, i) @ v for i in range(len(pairs))]).mean().item()


def direction_stats(vec, layer_idx, shift=None, pairs=None):
    v = unit_vec(vec)
    pairs = FIT_PAIRS if pairs is None else pairs
    p_neg = continuation_projection(pairs, layer_idx, v, "negative")
    p_pos = continuation_projection(pairs, layer_idx, v, "positive")
    measured = p_pos - p_neg
    return {
        "v": v, "layer": layer_idx, "p_neg": p_neg, "p_pos": p_pos,
        "shift": measured if shift is None else shift, "measured_shift": measured,
    }


def lam_for(stats, mag):
    return mag / stats["shift"] if abs(stats["shift"]) > 1e-6 else 0.0


def add_along(hidden, v, amount):
    vv = v.to(device=hidden.device, dtype=hidden.dtype)
    return hidden + amount * vv


def make_add_intervention(stats, lam, prefix_len, continuation_len):
    def intervention(hidden):
        patched = hidden.clone()
        pos = prediction_positions(prefix_len, continuation_len, hidden.device)
        patched[:, pos, :] = add_along(patched[:, pos, :], stats["v"], lam * stats["shift"])
        return patched
    return intervention


def score_stats(stats, lam, pairs=None):
    def factory(prefix_len, cont_len):
        return make_add_intervention(stats, lam, prefix_len, cont_len)
    return behavior_scores(pairs, layer_idx=stats["layer"], intervention_factory=factory)


def generate_steered(prompt, stats, lam, max_new_tokens=90, do_sample=True,
                     temperature=0.7, seed=0):
    def steer(hidden):
        patched = hidden.clone()
        patched[:, -1:, :] = add_along(patched[:, -1:, :], stats["v"], lam * stats["shift"])
        return patched

    inputs = chat_prefix_ids(prompt)
    gen = dict(**inputs, max_new_tokens=max_new_tokens, do_sample=do_sample,
               use_cache=True, pad_token_id=tokenizer.eos_token_id)
    if do_sample:
        gen.update(temperature=temperature, top_p=0.95)
        torch.manual_seed(seed)

    with layer_hook(model, stats["layer"], steer), torch.no_grad():
        out = model.generate(**gen)
    return tokenizer.decode(out[0, inputs["input_ids"].shape[-1]:], skip_special_tokens=True).strip()

In [14]:
# Residual mean-difference layer sweep.
# Find layers where a distributed residual direction moves behaviour.
REQUESTED_SWEEP_LAYERS = [4, 8, 12, 16, 20, 24, 28]
NUM_MODEL_LAYERS = len(model.model.layers)
SWEEP_LAYERS = [layer for layer in REQUESTED_SWEEP_LAYERS if layer < NUM_MODEL_LAYERS]
if NUM_MODEL_LAYERS - 1 not in SWEEP_LAYERS:
    SWEEP_LAYERS.append(NUM_MODEL_LAYERS - 1)
print(f"Model has {NUM_MODEL_LAYERS} layers; sweeping {SWEEP_LAYERS}")

# dose in multiples of each direction's measured shift
LAM_GRID = [-2.0, 0.5, 1.0, 2.0, 4.0, 8.0]
N_RANDOM = 3

residual_cache = {}


def capture_layer_residual(prompt, layer_idx):
    """layer_residual over a plain (non-chat) prompt, cached across the sweep."""
    key = (layer_idx, prompt)
    if key not in residual_cache:
        ids = tokenizer(prompt, return_tensors="pt")["input_ids"]
        residual_cache[key] = layer_residual(model, ids, layer_idx)
    return residual_cache[key]


def pooled_residual(prompt, layer_idx, pooling="mean"):
    # BOS carries an outlier-norm activation that would dominate a mean and win
    # any max, so it is dropped from every pooling mode.
    resid = capture_layer_residual(prompt, layer_idx)[1:]
    if pooling == "mean":
        return resid.mean(dim=0)
    if pooling == "final":
        return resid[-1]
    if pooling == "max_norm":
        return resid[resid.norm(dim=-1).argmax()]
    raise ValueError(f"unknown pooling mode: {pooling}")


def contrastive_mean_direction(positive_prompts, negative_prompts, layer_idx, pooling="mean"):
    pos = torch.stack([pooled_residual(p, layer_idx, pooling=pooling) for p in positive_prompts]).mean(dim=0)
    neg = torch.stack([pooled_residual(p, layer_idx, pooling=pooling) for p in negative_prompts]).mean(dim=0)
    return pos - neg


_rng = torch.Generator().manual_seed(11)

residual_stats = {}
random_stats = {}
residual_sweep_rows = []

DIRECTION_SPECS = [
    ("anger", angry_prompts, control_prompts),
    ("passive_aggressive", passive_aggressive_prompts, control_prompts[:len(passive_aggressive_prompts)]),
]

for direction_name, positive_prompts, negative_prompts in DIRECTION_SPECS:
    print("\n" + "#" * 80)
    print(f"residual direction: {direction_name}")
    for layer_idx in SWEEP_LAYERS:
        raw = contrastive_mean_direction(positive_prompts, negative_prompts, layer_idx)
        stats = direction_stats(raw, layer_idx)
        residual_stats[(direction_name, layer_idx)] = stats
        print(f"\nlayer {layer_idx}: p_neg={stats['p_neg']:+.3f} p_pos={stats['p_pos']:+.3f} "
              f"shift={stats['shift']:+.3f}")

        for lam in LAM_GRID:
            scores = score_stats(stats, lam, FIT_PAIRS)
            mean, delta = print_behavior_table(
                f"{direction_name} layer={layer_idx} lam={lam}", scores, reference=clean_fit,
            )
            residual_sweep_rows.append({
                "kind": "residual_mean", "direction": direction_name, "layer": layer_idx,
                "lam": lam, "mag": round(lam * stats["shift"], 2), "mean": mean, "delta": delta, "stats": stats,
            })

# Null control: arbitrary directions given the same displacement as the anger
# direction at that layer. Matching the displacement rather than the target matters,
# because activations already sit near zero projection on a random direction while
# p_pos is measured on the real direction itself.
for layer_idx in SWEEP_LAYERS:
    ref_shift = residual_stats[("anger", layer_idx)]["shift"]
    for i in range(N_RANDOM):
        raw = torch.randn(model.config.hidden_size, generator=_rng)
        stats = direction_stats(raw, layer_idx, shift=ref_shift)
        random_stats[(layer_idx, i)] = stats
        for lam in LAM_GRID:
            mean, delta = summarize_scores(score_stats(stats, lam, FIT_PAIRS), reference=clean_fit)
            residual_sweep_rows.append({
                "kind": "random", "direction": f"random{i}", "layer": layer_idx,
                "lam": lam, "mag": round(lam * stats["shift"], 2), "mean": mean, "delta": delta, "stats": stats,
            })

residual_sweep_rows = sorted(residual_sweep_rows, key=lambda row: row["delta"], reverse=True)
random_deltas = [r["delta"] for r in residual_sweep_rows if r["kind"] == "random"]
NULL_P95 = float(np.percentile(random_deltas, 95))
print("\n" + "#" * 80)
print(f"random-direction deltas: mean {np.mean(random_deltas):+.4f} p95 {NULL_P95:+.4f} max {max(random_deltas):+.4f}")
print("Top residual mean-difference interventions (fit split)")
for row in [r for r in residual_sweep_rows if r["kind"] == "residual_mean"][:12]:
    print(
        f"{row['direction']:>18} layer {row['layer']:>2} |dx| {row['mag']:>5} "
        f"mean {row['mean']:+.4f} delta {row['delta']:+.4f} "
        f"{'(above null p95)' if row['delta'] > NULL_P95 else '(within null)'}"
    )

TOP_LAYER_KEYS = []
for row in residual_sweep_rows:
    if row["kind"] != "residual_mean":
        continue
    key = (row["direction"], row["layer"])
    if key not in TOP_LAYER_KEYS:
        TOP_LAYER_KEYS.append(key)
    if len(TOP_LAYER_KEYS) >= 3:
        break
print("Selected layer/direction keys for SAE tests:", TOP_LAYER_KEYS)

Model has 24 layers; sweeping [4, 8, 12, 16, 20, 23]

################################################################################
residual direction: anger

layer 4: p_neg=+0.175 p_pos=+0.351 shift=+0.176

anger layer=4 lam=-2.0
mean behavioral logit-diff: -0.5540  mean_delta -0.0393 [-0.1030, +0.0245]
  pair 0: -0.5903  delta +0.0118
  pair 1: -0.7506  delta -0.1264
  pair 2: -1.0273  delta -0.1247
  pair 3: -0.3083  delta +0.0558
  pair 4: -0.0935  delta -0.0130

anger layer=4 lam=0.5
mean behavioral logit-diff: -0.5015  mean_delta +0.0132 [-0.0093, +0.0357]
  pair 0: -0.6206  delta -0.0185
  pair 1: -0.5740  delta +0.0502
  pair 2: -0.8819  delta +0.0207
  pair 3: -0.3792  delta -0.0151
  pair 4: -0.0517  delta +0.0288

anger layer=4 lam=1.0
mean behavioral logit-diff: -0.4851  mean_delta +0.0296 [-0.0017, +0.0621]
  pair 0: -0.6244  delta -0.0222
  pair 1: -0.5382  delta +0.0859
  pair 2: -0.8531  delta +0.0494
  pair 3: -0.3606  delta +0.0035
  pair 4: -0.0491  delta +0.0315


In [15]:
# SAE arms: a group direction built from the top features, and the features alone.
SAE_TOP_N = 12
SAE_GROUP_N = 8
SELECTIVITY = 3.0
sae_cache = {LAYER: sae} if "sae" in globals() else {}
sae_feature_rows = []


def load_sae_for_layer(layer_idx):
    if layer_idx not in sae_cache:
        layer_device = next(model.model.layers[layer_idx].parameters()).device
        layer_sae = SAE.from_pretrained(
            release=SAE_RELEASE, sae_id=f"layer{layer_idx}",
            device=str(layer_device), dtype="float32",
        )
        layer_sae.eval()
        sae_cache[layer_idx] = layer_sae
    return sae_cache[layer_idx]


def pooled_sae_features(prompt, layer_idx, layer_sae, pooling="max"):
    resid = capture_layer_residual(prompt, layer_idx)[1:].to(layer_sae.W_enc.device, dtype=layer_sae.W_enc.dtype)
    feats = layer_sae.encode(resid)
    if pooling == "max":
        return feats.amax(dim=0).detach().cpu()
    if pooling == "mean":
        return feats.mean(dim=0).detach().cpu()
    if pooling == "final":
        return feats[-1].detach().cpu()
    raise ValueError(f"unknown pooling mode: {pooling}")


def score_sae_features(layer_idx, positive_prompts, negative_prompts, pooling="max", top_n=SAE_TOP_N):
    """Rank by effect size, not raw activation delta.

    Feature activation scales differ by orders of magnitude across the dictionary,
    so a raw mean difference ranks loud features above selective ones. Ranking by
    a pooled-SD-normalised delta with a selectivity floor keeps the vector
    weighting in activation units where it belongs.
    """
    layer_sae = load_sae_for_layer(layer_idx)
    pos = torch.stack([pooled_sae_features(p, layer_idx, layer_sae, pooling=pooling) for p in positive_prompts])
    neg = torch.stack([pooled_sae_features(p, layer_idx, layer_sae, pooling=pooling) for p in negative_prompts])
    diff = pos.mean(dim=0) - neg.mean(dim=0)
    sd = ((pos.var(dim=0) + neg.var(dim=0)) / 2).clamp_min(1e-8).sqrt()
    effect = diff / sd
    selective = pos.mean(dim=0) > SELECTIVITY * (neg.mean(dim=0) + 1e-6)
    effect = torch.where(selective & (diff > 0), effect, torch.full_like(effect, -float("inf")))
    vals, ids = effect.topk(top_n)
    return layer_sae, [
        {
            "feature_id": int(feature_id),
            "effect": float(value),
            "score": float(diff[feature_id]),
            "positive_mean": float(pos[:, feature_id].mean()),
            "negative_mean": float(neg[:, feature_id].mean()),
        }
        for value, feature_id in zip(vals, ids)
        if torch.isfinite(value)
    ]


def weighted_feature_group(layer_sae, feature_rows, group_n=SAE_GROUP_N):
    vec = None
    for row in feature_rows[:group_n]:
        part = row["score"] * layer_sae.W_dec[row["feature_id"]].detach().float().cpu()
        vec = part if vec is None else vec + part
    return vec


for direction_name, layer_idx in TOP_LAYER_KEYS:
    positive_prompts = angry_prompts if direction_name == "anger" else passive_aggressive_prompts
    negative_prompts = control_prompts[:len(positive_prompts)]
    layer_sae, feature_rows = score_sae_features(layer_idx, positive_prompts, negative_prompts)

    print("\n" + "#" * 80)
    print(f"SAE candidates for {direction_name} layer {layer_idx}")
    for row in feature_rows:
        print(
            f"feature {row['feature_id']:>6} effect {row['effect']:+.2f} delta {row['score']:+.3f} "
            f"pos {row['positive_mean']:.3f} ctrl {row['negative_mean']:.3f}"
        )

    group_ids = [row["feature_id"] for row in feature_rows[:SAE_GROUP_N]]
    gstats = direction_stats(weighted_feature_group(layer_sae, feature_rows), layer_idx)
    print(f"group of {len(group_ids)}: shift={gstats['shift']:+.3f}")
    for lam in LAM_GRID:
        scores = score_stats(gstats, lam, FIT_PAIRS)
        mean, delta = print_behavior_table(
            f"SAE group {direction_name} layer={layer_idx} lam={lam}", scores, reference=clean_fit,
        )
        sae_feature_rows.append({
            "kind": "sae_group", "direction": direction_name, "layer": layer_idx,
            "lam": lam, "mag": round(lam * gstats["shift"], 2), "mean": mean, "delta": delta,
            "stats": gstats, "features": group_ids,
        })

    for row in feature_rows[:3]:
        fstats = direction_stats(layer_sae.W_dec[row["feature_id"]], layer_idx)
        for lam in LAM_GRID:
            mean, delta = summarize_scores(score_stats(fstats, lam, FIT_PAIRS), reference=clean_fit)
            sae_feature_rows.append({
                "kind": "single_feature", "direction": direction_name, "layer": layer_idx,
                "feature_id": row["feature_id"], "lam": lam, "mag": round(lam * fstats["shift"], 2),
                "mean": mean, "delta": delta, "stats": fstats,
            })
        print(
            f"single feature quick test {row['feature_id']} (shift {fstats['shift']:+.3f}): "
            + ", ".join(
                f"|dx| {r['mag']} delta {r['delta']:+.4f}"
                for r in sae_feature_rows
                if r.get("feature_id") == row["feature_id"] and r["layer"] == layer_idx
            )
        )

sae_feature_rows = sorted(sae_feature_rows, key=lambda row: row["delta"], reverse=True)
print("\n" + "#" * 80)
print(f"Top SAE interventions (fit split; random-direction p95 = {NULL_P95:+.4f})")
for row in sae_feature_rows[:12]:
    label = (f"group {row['features'][:4]}..." if row["kind"] == "sae_group"
             else f"feature {row['feature_id']}")
    print(
        f"{row['kind']:>14} {row['direction']:>18} layer {row['layer']:>2} "
        f"|dx| {row['mag']:>5} mean {row['mean']:+.4f} delta {row['delta']:+.4f} "
        f"{'above null' if row['delta'] > NULL_P95 else 'within null'} {label}"
    )


################################################################################
SAE candidates for anger layer 8
feature  23501 effect +2.41 delta +0.208 pos 0.208 ctrl 0.000
feature  21872 effect +2.35 delta +0.164 pos 0.164 ctrl 0.000
feature  19766 effect +2.24 delta +0.190 pos 0.219 ctrl 0.029
feature   2826 effect +2.08 delta +0.094 pos 0.094 ctrl 0.000
feature  15474 effect +1.98 delta +0.231 pos 0.239 ctrl 0.009
feature      2 effect +1.93 delta +0.188 pos 0.208 ctrl 0.020
feature  12576 effect +1.91 delta +0.460 pos 0.479 ctrl 0.019
feature  18846 effect +1.90 delta +0.177 pos 0.236 ctrl 0.059
feature  12573 effect +1.83 delta +0.133 pos 0.140 ctrl 0.008
feature   4762 effect +1.80 delta +0.492 pos 0.547 ctrl 0.055
feature   5014 effect +1.73 delta +0.268 pos 0.268 ctrl 0.000
feature  21252 effect +1.72 delta +0.738 pos 0.913 ctrl 0.174
group of 8: shift=+0.201

SAE group anger layer=8 lam=-2.0
mean behavioral logit-diff: -0.5564  mean_delta -0.0417 [-0.1041, +0.0205]
  pair 

## SAE Steering Retry with higher scalars


In [16]:
MAG_GRID_ABS = [1.6, 4.0, 8.0, 12.0, 16.0, 24.0]

fair_rows = []
for direction_name, layer_idx in TOP_LAYER_KEYS:
    positive_prompts = angry_prompts if direction_name == "anger" else passive_aggressive_prompts
    negative_prompts = control_prompts[:len(positive_prompts)]
    layer_sae, feature_rows = score_sae_features(layer_idx, positive_prompts, negative_prompts)

    arms = [("sae_group_absdose", None, direction_stats(weighted_feature_group(layer_sae, feature_rows), layer_idx))]
    arms += [
        ("single_feature", row["feature_id"], direction_stats(layer_sae.W_dec[row["feature_id"]], layer_idx))
        for row in feature_rows[:3]
    ]
    top_resid = next(r for r in residual_sweep_rows if r["kind"] == "residual_mean" and r["direction"] == direction_name)
    arms.append(("residual_absdose", None, top_resid["stats"]))

    for kind, feature_id, stats in arms:
        for mag in MAG_GRID_ABS:
            lam = lam_for(stats, mag)
            mean, delta = summarize_scores(score_stats(stats, lam, FIT_PAIRS), reference=clean_fit)
            fair_rows.append({
                "kind": kind, "direction": direction_name, "layer": stats["layer"],
                "feature_id": feature_id, "lam": round(lam, 2), "mag": mag,
                "mean": mean, "delta": delta, "stats": stats,
            })
            flag = "above null" if delta > NULL_P95 else "within null"
            label = f"feature {feature_id}" if feature_id is not None else kind
            print(f"{kind:>17} {direction_name:>18} layer {stats['layer']:>2} "
                  f"|dx| {mag:>5} lam {fair_rows[-1]['lam']:>9} delta {delta:+.4f} {flag} {label}")

fair_rows = sorted(fair_rows, key=lambda row: row["delta"], reverse=True)
top = fair_rows[0]
print(f"\nbest fair-dose arm: {top['kind']} feature={top['feature_id']} "
      f"layer {top['layer']} |dx| {top['mag']} delta {top['delta']:+.4f}")

sae_group_absdose              anger layer  8 |dx|   1.6 lam      7.95 delta +0.1982 above null sae_group_absdose
sae_group_absdose              anger layer  8 |dx|   4.0 lam     19.88 delta +0.4626 above null sae_group_absdose
sae_group_absdose              anger layer  8 |dx|   8.0 lam     39.76 delta +0.5384 above null sae_group_absdose
sae_group_absdose              anger layer  8 |dx|  12.0 lam     59.64 delta +0.5309 above null sae_group_absdose
sae_group_absdose              anger layer  8 |dx|  16.0 lam     79.52 delta +0.4162 above null sae_group_absdose
sae_group_absdose              anger layer  8 |dx|  24.0 lam    119.28 delta +0.3044 above null sae_group_absdose
   single_feature              anger layer  8 |dx|   1.6 lam      13.0 delta +0.3079 above null feature 23501
   single_feature              anger layer  8 |dx|   4.0 lam     32.51 delta +0.3892 above null feature 23501
   single_feature              anger layer  8 |dx|   8.0 lam     65.01 delta +0.3957 above null 

In [17]:
import textwrap

GEN_MAX_NEW_TOKENS = 512
GEN_TEMPERATURE = 0.7
N_SHOW = 8

probe_prompts = [pair["prompt"] for pair in FIT_PAIRS[:3]]

def candidate_name(row):
    tail = f"feature {row['feature_id']}" if row["kind"] == "single_feature" else row["direction"]
    return f"{row['kind']} {tail} L{row['layer']} lam={row['lam']:.1f}"

def magnitude(row):
    return abs(row["mag"])

def show(tag, text):
    print(f"\n  --- {tag} ---")
    print(textwrap.indent(text.strip(), "      "))

def is_coherent(text):
    """Heavy steering can collapse text into fragments or loops that still game
    the teacher-forced metric; a winning candidate must also read as language."""
    words = text.lower().split()
    if len(words) < 5 or "\ufffd" in text:
        return False
    trigrams = [tuple(words[j:j + 3]) for j in range(len(words) - 2)]
    repeat = 1 - len(set(trigrams)) / max(len(trigrams), 1)
    return len(set(words)) / len(words) > 0.5 and repeat < 0.2

all_rows = sorted(
    [r for r in residual_sweep_rows + sae_feature_rows + fair_rows if r["delta"] is not None],
    key=lambda r: r["delta"], reverse=True,
)
real = [r for r in all_rows if r["kind"] != "random"]
rand = [r for r in all_rows if r["kind"] == "random"]

print(f"random-direction null: mean {np.mean([r['delta'] for r in rand]):+.4f}  p95 {NULL_P95:+.4f}")
print("\ntop candidates on the fit split")
for row in real[:N_SHOW]:
    flag = "" if row["delta"] > NULL_P95 else "   (within null)"
    print(f"  delta {row['delta']:+.4f}  logit-diff {row['mean']:+.4f}  "
          f"|dx| {magnitude(row):5.2f}  lam {row['lam']:6.2f}  {candidate_name(row)}{flag}")

# Coherence-gated selection: walk the metric ranking and keep the first candidate
# whose steered generations still read as language.
best_random = rand[0]
best = None
for row in real:
    outs = [generate_steered(p, row["stats"], row["lam"], max_new_tokens=GEN_MAX_NEW_TOKENS,
                             temperature=GEN_TEMPERATURE, seed=i)
            for i, p in enumerate(probe_prompts[:2])]
    if all(is_coherent(o) for o in outs):
        best = row
        break
    print(f"  rejected {candidate_name(row)}: steered generations incoherent")
assert best is not None, "no candidate passed the coherence gate"
print(f"\nselected: {candidate_name(best)} (lam {best['lam']:.2f}, coherence-gated)")

for label, pairs, clean in [
    ("test pairs", TEST_PAIRS, clean_test),
    ("lexical control", lexical_pairs, clean_lexical),
]:
    scores = score_stats(best["stats"], best["lam"], pairs)
    mean, lo, hi = boot_ci([s - c for s, c in zip(scores, clean)])
    before, after = sum(clean) / len(clean), sum(scores) / len(scores)
    print(f"{label:>16}: logit-diff {before:+.4f} -> {after:+.4f}   delta {mean:+.4f} [{lo:+.4f}, {hi:+.4f}]")

for i, prompt in enumerate(probe_prompts):
    print("\n" + "=" * 80)
    print(f"PROMPT: {prompt}")
    for tag, stats, lam in [
        ("unsteered", best["stats"], 0.0),
        (f"steered: {candidate_name(best)}, |dx| {magnitude(best):.2f}", best["stats"], best["lam"]),
        (f"random direction, |dx| {magnitude(best_random):.2f}", best_random["stats"], best_random["lam"]),
    ]:
        show(tag, generate_steered(prompt, stats, lam, max_new_tokens=GEN_MAX_NEW_TOKENS,
                                   temperature=GEN_TEMPERATURE, seed=i))

random-direction null: mean +0.0000  p95 +0.0701

top candidates on the fit split
  delta +0.7619  logit-diff +0.2472  |dx| 16.00  lam  52.24  residual_absdose anger L8 lam=52.2
  delta +0.7619  logit-diff +0.2472  |dx| 16.00  lam  52.24  residual_absdose anger L8 lam=52.2
  delta +0.7540  logit-diff +0.2393  |dx| 24.00  lam  27.12  residual_absdose passive_aggressive L16 lam=27.1
  delta +0.7390  logit-diff +0.2244  |dx| 12.00  lam  39.18  residual_absdose anger L8 lam=39.2
  delta +0.7390  logit-diff +0.2244  |dx| 12.00  lam  39.18  residual_absdose anger L8 lam=39.2
  delta +0.6737  logit-diff +0.1590  |dx| 24.00  lam 1284.07  single_feature feature 21872 L8 lam=1284.1
  delta +0.6412  logit-diff +0.1265  |dx| 24.00  lam  78.36  residual_absdose anger L8 lam=78.4
  delta +0.6412  logit-diff +0.1265  |dx| 24.00  lam  78.36  residual_absdose anger L8 lam=78.4

selected: residual_absdose anger L8 lam=52.2 (lam 52.24, coherence-gated)
      test pairs: logit-diff -0.9079 -> +0.4844   de